# RENDU INFO TC1

## Objectif du devoir

Le but de ce devoir est de **déterminer automatiquement une palette de couleurs optimale** pour une image donnée. Cette palette devra valider les contraintes suivantes : 

1. utiliser moins de couleurs que le nombre disponible dans l'image donnée;
2. être la plus représentative possible des couleurs de l'image donnée. 

Comme nous l'avons vu dans le TD 4, les couleurs peuvent être encodée par composantes rouge, verte et bleue (soit 256 valeurs possibles par composante, autrement dit sur 8 bits) ainsi potentiellement utiliser $256 \times 256 \times 256 = 16 777 216$ couleurs. En réalité, beaucoup moins sont nécessaires et surtout perceptibles par l'humain. Réduire le nombre de couleurs ou réaliser une "_quantification de couleurs_" est une tâche fréquente et c'est une fonctionnalité classique des outils éditeurs d'images (Photoshop, Gimp, etc.) implémentée aussi dans le module Pillow de Python. A noter que cette réduction s'effectue avec perte de couleurs et doit être réalisée avec les bons paramètres (nombre et choix des couleurs) ce qui est votre objectif. 

La figure ci-dessous illustre le problème à résoudre : étant donnée une image en entrée, proposer une liste de couleurs (que l'on appellera la palette), afin de re-colorier une image en sortie.

<div style="text-align:center;">
<table>
  <tr>
    <td>
      <img src="figures/color-rainbow.png" alt="Image originale" style="height:5cm;">
      <p>Image donnée</p>
    </td>
    <td>
      <img src="figures/rainbow-palette-8.png" alt="Palette de 8 couleurs représentatives" style="height:5cm;">
      <p>Palette de 8 couleurs représentatives</p>
    </td>
    <td>
      <img src="figures/rainbow-recoloriee.png" alt="Image originale recoloriée avec la palette" style="height:5cm;">
      <p>Image donnée recoloriée avec la palette</p>
    </td>
  </tr>
</table>
</div>

## Étapes de travail

Voici des étapes de travail suggérées :

1. Prenez une image de votre choix (pas trop grande) en la chargeant avec PIL. Lister les couleurs présentes, identifier celles qui sont uniques et leur fréquence.

In [ ]:
from PIL import Image
from IPython.display import display

# On reprend une fonction du TD4 pour générer des images de test rapidement.
def setRegion(x, y, w, h, color, px):
    for i in range(int(x), int(x + w)):
        for j in range(int(y), int(y + h)):
            px[i, j] = color

def lister_couleurs(im):
    """
    Retourne un dictionnaire {couleur: fréquence} pour tous les pixels de l'image.
    Arguments:
        im (Image): image PIL en mode RGB
    Renvoie :
        dict: {(r,g,b): compteur}
    """
    px = im.load()
    W, H = im.size
    freq = {}
    for x in range(W):
        for y in range(H):
            c = px[x, y]
            if c in freq:
                freq[c] += 1
            else:
                freq[c] = 1
    return freq

In [ ]:
# Test sur une image simple : image noire avec un carré blanc
W, H = 100, 100
im_test = Image.new('RGB', (W, H))
px_test = im_test.load()
setRegion(W//3, H//3, W//3, H//3, (255, 255, 255), px_test)
display(im_test)

freq_test = lister_couleurs(im_test)
print("Nombre de couleurs :", len(freq_test))
liste_couples = []
for couleur, nombre in freq_test.items():
    liste_couples.append((nombre, couleur))

liste_couples.sort(reverse=True)

for n, c in liste_couples:
    print(f"{c} : {n} pixels")

assert len(freq_test) == 2, "Il doit y avoir exactement 2 couleurs"

In [ ]:
# Test sur une image plus complexe
im = Image.open("lyon.png").convert("RGB")
freq = lister_couleurs(im)
print(f"Taille image : {im.size}")
print(f"Nombre de couleurs : {len(freq)}")
liste_couples = []
for couleur, nombre in freq.items():
    liste_couples.append((nombre, couleur))

liste_couples.sort(reverse=True)
top10 = liste_couples[:10]
print("Top 10 couleurs les plus fréquentes :")
for n, c in top10:
    print(f"{c} : {n} pixels")
display(im)

2. Proposez une méthode (naïve pour commencer) de choix d'une palette de $k$ couleurs. Affichez là sous forme d'image (exemple de d'image au milieu de la figure du dessus) avec une nouvelle image PIL. Utilisez également des images simples où le résultat attendu est connu comme pour les images ci-dessous :

  <div style="text-align:center;">
    <table>
      <tr>
        <td>
          <img src="figures/1-color-back.png" alt="1 couleur noir" style="width:3cm;">
          <p>1 couleur noir</p>
        </td>
        <td>
          <img src="figures/4-color.png" alt="4 couleurs" style="width:3cm;">
          <p>4 couleurs</p>
        </td>
      </tr>
    </table>
  </div>

In [ ]:
def palette_naive(freq, k):
    """
    Retourne une palette de k couleurs = les k couleurs les plus fréquentes.
    Arguments :
        freq (dict): dictionnaire {couleur: fréquence}
        k (int): nombre de couleurs souhaitées dans la palette
    Renvoie :
        list: liste de k tuples (r, g, b) (palette)
    """
    # Trier par fréquence décroissante et prendre les k premières
    liste_couples = []
    for couleur, freq in freq.items():
        liste_couples.append((freq, couleur))

    liste_couples.sort(reverse=True)
    k_couples = liste_couples[:k]
    return [c for c, _ in k_couples]

In [ ]:
def afficher_palette(palette, taille_case=50):
    """
    Crée et affiche une image PIL montrant les couleurs de la palette.
    Args:
        palette (list): liste de tuples (r, g, b)
        taille_case (int): taille en pixels de chaque case de couleur
    Returns:
        Image: image PIL de la palette
    """
    k = len(palette)
    im_palette = Image.new('RGB', (k * taille_case, taille_case))
    px_p = im_palette.load()
    for i in range(len(palette)):
        couleur = palette[i]
        setRegion(i * taille_case, 0, taille_case, taille_case, couleur, px_p)
    return im_palette

# Test sur image 1 couleur (noire)
im_1 = Image.new('RGB', (50, 50))  # image entièrement noire
freq_1 = lister_couleurs(im_1)
palette_1 = palette_naive(freq_1, 1)
print("Palette image noire (k=1):", palette_1)
assert palette_1 == [(0, 0, 0)], "Doit retourner [(0,0,0)]"
display(afficher_palette(palette_1))

# Test sur image 4 couleurs (rouge, vert, bleu, jaune)
im_4 = Image.new('RGB', (100, 100))
px_4 = im_4.load()
setRegion(0, 0, 50, 50, (255, 0, 0), px_4) # rouge
setRegion(50, 0, 50, 50, (0, 255, 0), px_4) # vert
setRegion(0, 50, 50, 50, (0, 0, 255), px_4) # bleu
setRegion(50, 50, 50, 50, (255, 255, 0), px_4) # jaune

freq_4 = lister_couleurs(im_4)
palette_4 = palette_naive(freq_4, 4)
print("Palette image 4 couleurs (k=4):", sorted(palette_4))
assert len(palette_4) == 4
display(afficher_palette(palette_4))
display(im_4)

# Test sur une image plus complexe
im = Image.open("lyon.png").convert("RGB")
freq = lister_couleurs(im)
palette = palette_naive(freq, 8)
print("Palette image 8 couleurs (k=8):", sorted(palette))
assert len(palette) == 8
display(afficher_palette(palette))
display(im)

3. Re-coloriez une image avec une palette de $k$ couleurs, et affichez le résultat sous forme d'image PIL. Pour re-colorier chaque pixel, prendre la couleur la plus proche dans la palette en utilisant une fonction de distance (Euclidienne par exemple..).

In [ ]:
from math import sqrt
def distance(c1, c2):
    return sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2 + (c1[2]-c2[2])**2)

def couleur_la_plus_proche(c, palette):
    """
    Retourne la couleur de la palette la plus proche de c (distance euclidienne).
    Arguments :
        c (tuple): couleur (r, g, b)
        palette (list): liste de tuples (r, g, b)
    Renvoie :
        tuple: couleur la plus proche dans la palette
    """
    plus_proche = palette[0]
    dist_min = distance(c, palette[0])
    for couleur in palette[1:]:
        d = distance(c, couleur)
        if d < dist_min:
            dist_min = d
            plus_proche = couleur
    return plus_proche

def recolorier(im, palette):
    """
    Re-colorier chaque pixel de l'image avec la couleur la plus proche dans la palette.
    Arguments :
        im (Image): image PIL originale
        palette (list): liste de tuples (r, g, b)
    Renvoie :
        Image: nouvelle image re-coloriée
    """
    W, H = im.size
    im_out = Image.new('RGB', (W, H))
    px_in = im.load()
    px_out = im_out.load()
    for x in range(W):
        for y in range(H):
            c = px_in[x, y]
            c_proche = couleur_la_plus_proche(c, palette)
            px_out[x, y] = c_proche
    return im_out

In [ ]:
# Test 1
# On commence par tester uniquement le recoloriage sur l'image 4 couleurs, avec une palette réduite à 2 couleurs.
# On prend rouge et bleu comme palette, vert et jaune iront vers le plus proche
palette_reduite = [(255, 0, 0), (0, 0, 255)]
im_recoloriee = recolorier(im_4, palette_reduite)
print("Re-coloriage avec 2 couleurs (rouge, bleu) :")
display(im_4)
display(im_recoloriee)

# Test 2 : image 4 couleurs re-coloriée avec palette
palette_4 = palette_naive(freq_4, 3) #Palette de 3 couleurs
im_4_recoloriee = recolorier(im_4, palette_4)
freq_exact = lister_couleurs(im_4_recoloriee)
assert len(freq_exact) == 3, "Doit avoir exactement 3 couleurs"
print("Test image 4 couleurs avec palette 3 couleurs réussi")

# Test 3 : sur une image plus complexe
im = Image.open("lyon.png").convert("RGB")
freq = lister_couleurs(im)
palette = palette_naive(freq, 8)
im_recoloriee = recolorier(im, palette)
display(afficher_palette(palette))
display(im_recoloriee)

La palette ne représente pas l'ensemble des couleurs : elle ne comprend que les couleurs les plus présentes, qui sont toutes concentrées dans le ciel.


4. Proposez une méthode de validation de votre approche. Par exemple affichez la différence entre l'image originale et celle re-coloriée. Calculez un score global d'erreur.


In [ ]:
def image_difference(im_orig, im_recoloriee):
    """
    Crée une image montrant la différence pixel par pixel entre deux images.
    Arguments :
        im_orig (Image): image originale
        im_recoloriee (Image): image re-coloriée
    Renvoie :
        Image: image de différence
    """
    W, H = im_orig.size
    im_diff = Image.new('RGB', (W, H))
    px_o = im_orig.load()
    px_r = im_recoloriee.load()
    px_d = im_diff.load()
    for x in range(W):
        for y in range(H):
            ro, go, bo = px_o[x, y]
            rr, gr, br = px_r[x, y]
            # Amplifier la différence pour la visualisation, en mettant un max à 255
            diff_r = min(255, abs(ro - rr) * 3)
            diff_g = min(255, abs(go - gr) * 3)
            diff_b = min(255, abs(bo - br) * 3)
            px_d[x, y] = (diff_r, diff_g, diff_b)
    return im_diff


def score_erreur(im_orig, im_recoloriee):
    """
    Calcule le score d'erreur global moyen entre l'image originale et re-coloriée.
    Score = moyenne des distances euclidiennes pixel par pixel.
    Arguments :
        im_orig (Image): image originale
        im_recoloriee (Image): image re-coloriée
    Renvoie :
        float: erreur moyenne
    """
    W, H = im_orig.size
    px_o = im_orig.load()
    px_r = im_recoloriee.load()
    total = 0.0
    for x in range(W):
        for y in range(H):
            c_o = px_o[x, y]
            c_r = px_r[x, y]
            total += distance(c_o, c_r)
    return total / (W * H)

In [ ]:
# Test : image 4 couleurs re-coloriée avec palette 3 couleurs
erreur_4 = score_erreur(im_4, im_4_recoloriee)
print(f"Erreur image 4 couleurs avec palette 3 couleurs : {erreur_4:.4f}")
print("Image de différence :")
display(image_difference(im_4, im_4_recoloriee))

# Test sur une image plus complexe
erreur_lyon = score_erreur(im, im_recoloriee)
print(f"Erreur image lyon avec palette 8 couleurs : {erreur_lyon:.4f}")
print("Image de différence :")
display(image_difference(im, im_recoloriee))

print("Tests étape 4 réussis")


5. Améliorez le choix des $k$ couleurs afin de minimiser l'erreur entre l'image originale et re-coloriée. Une piste possible est de trier les couleurs dans une liste, diviser cette liste en $k$ intervals de couleurs et prendre la couleur du milieu de chaque interval. D'autres méthodes plus avancées peuvent être explorées !


In [ ]:
def palette_intervalles(freq, k):
    """
    Choisit k couleurs en triant toutes les couleurs (par luminosité approximative r+g+b),
    en divisant en k intervalles et en prenant la couleur du milieu de chaque intervalle.
    Arguments :
        freq (dict): dictionnaire {couleur: fréquence}
        k (int): nombre de couleurs souhaitées
    Renvoie :
        list: liste de k tuples (r, g, b)
    """
    # On récupère toutes les couleurs (les clés du dictionnaire)
    couleurs = list(freq.keys())
    
    # Calcul du score de sensibilité pour chaque couleur
    liste_couples_score = []
    for c in couleurs:
        # Formule de luminance : 0.3*R + 0.59*G + 0.11*B
        score = 0.3 * c[0] + 0.59 * c[1] + 0.11 * c[2]
        liste_couples_score.append((score, c))

    # Tri par score (du plus sombre au plus clair)
    liste_couples_score.sort()

    # On reconstruit la liste des couleurs triées
    couleurs_triees = []
    for score, c in liste_couples_score:
        couleurs_triees.append(c)

    # Sélection des k couleurs réparties uniformément
    palette_opti = []
    n = len(couleurs_triees)
    
    for i in range(k):
        # On calcule l'indice du milieu de chaque paquet
        milieu_paquet = int((i + 0.5) * n / k)
        palette_opti.append(couleurs_triees[milieu_paquet])
        
    return palette_opti

In [ ]:
#Autre méthode : K-means

def calculer_moyenne(groupe):
    somme_r = 0
    somme_g = 0
    somme_b = 0
    for c in groupe: # c est un tuple (R, G, B)
        somme_r += c[0]
        somme_g += c[1]
        somme_b += c[2]
        
    n = len(groupe)
    return (somme_r // n, somme_g // n, somme_b // n)

def palette_kmeans(freq, k, iterations=5):
    """
    Calcule une palette de k couleurs représentatives en utilisant l'algorithme des K-means.
    Arguments :
        freq (dict): dictionnaire {couleur: fréquence}
        k (int): nombre de couleurs souhaitées
        iterations (int): nombre de fois que l'algorithme va affiner les centres
    Renvoie :
        list: liste de k tuples (r, g, b)
    """
    couleurs_uniques = list(freq.keys())
    n = len(couleurs_uniques)
    
    if n == 0:
        return []

    # Initialisation : on prend des couleurs réparties dans la liste
    centres = [couleurs_uniques[i * n // k] for i in range(k)]
    
    for _ in range(iterations):
        groupes = [[] for _ in range(k)]
        
        # ÉTAPE D'ASSIGNATION
        for c in couleurs_uniques:
            index_proche = 0
            dist_min = distance(c, centres[0])
            
            for i in range(1, k):
                d = distance(c, centres[i])
                if d < dist_min:
                    dist_min = d
                    index_proche = i
            
            groupes[index_proche].append(c)
            
        # ÉTAPE DE MISE À JOUR
        nouveaux_centres = []
        for i in range(k):
            if groupes[i]:
                # On calcule la couleur moyenne du groupe
                nouveaux_centres.append(calculer_moyenne(groupes[i]))
            else:
                nouveaux_centres.append(couleurs_uniques[i * n // k])
        
        centres = nouveaux_centres
                
    return centres

In [ ]:
# Test 1 : image 4 couleurs re-coloriée avec palette (Intervalles)
palette_4 = palette_intervalles(freq_4, 3) #Palette de 3 couleurs
im_4_recoloriee = recolorier(im_4, palette_4)
freq_exact = lister_couleurs(im_4_recoloriee)
assert len(freq_exact) == 3, "Doit avoir exactement 3 couleurs"
print("Test image 4 couleurs avec palette 3 couleurs réussi")

# Test 2 : sur une image plus complexe (Intervalles)
im = Image.open("lyon.png").convert("RGB")
freq = lister_couleurs(im)
palette = palette_intervalles(freq, 8)
im_recoloriee = recolorier(im, palette)
display(afficher_palette(palette))
display(im_recoloriee)

# Test 3 : image 4 couleurs re-coloriée avec palette (K-means)
palette_4 = palette_kmeans(freq_4, 3) #Palette de 3 couleurs
im_4_recoloriee = recolorier(im_4, palette_4)
freq_exact = lister_couleurs(im_4_recoloriee)
assert len(freq_exact) == 3, "Doit avoir exactement 3 couleurs"
print("Test image 4 couleurs avec palette 3 couleurs réussi")

# Test 4 : sur une image plus complexe (K-means)
im = Image.open("lyon.png").convert("RGB")
freq = lister_couleurs(im)
palette = palette_kmeans(freq, 8)
im_recoloriee = recolorier(im, palette)
display(afficher_palette(palette))
display(im_recoloriee)


6. Testez votre palette sur plusieurs images de votre choix ou générées automatiquement avec un nombre et une distribution connue de couleurs. Comparer les performances de vos techniques avec d'autres méthodes (cette fois vous pouvez utiliser un éditeur de texte ou la fonction _quantize_ de PIL [(doc)](https://pillow.readthedocs.io/en/stable/reference/Image.html).


In [ ]:
def palette_depuis_quantize(im, k):
    """
    Utilise la fonction quantize de PIL pour obtenir une palette de k couleurs.
    Arguments :
        im (Image): image PIL en mode RGB
        k (int): nombre de couleurs
    Renvoie :
        list: liste de k tuples (r, g, b)
    """
    im_q = im.quantize(colors=k)
    # Extraire la palette de l'image
    palette_brute = im_q.getpalette()  # liste en [r,g,b, r,g,b, ...]
    palette = []
    for i in range(k):
        r, g, b = palette_brute[3*i], palette_brute[3*i+1], palette_brute[3*i+2]
        palette.append((r, g, b))
    return palette

In [ ]:
def comparer_methodes(im, k):
    """
    Compare les trois méthodes de palettes sur une image donnée.
    Affiche les palettes et les scores d'erreur.
    Args:
        im (Image): image PIL
        k (int): nombre de couleurs
    """
    freq = lister_couleurs(im)
    print(f"Comparaison pour k={k} couleurs")
    print(f"Nombre de couleurs dans l'image originale : {len(freq)}")
    
    methodes = [
        ("Naïve", palette_naive(freq, k)),
        ("Intervalles médiane", palette_intervalles(freq, k)),
        ("K-means", palette_kmeans(freq, k)),
        ("PIL quantize", palette_depuis_quantize(im, k)),
    ]
    
    for nom, palette in methodes:
        im_r = recolorier(im, palette)
        erreur = score_erreur(im, im_r)
        print(f"  {nom} : erreur = {erreur:.4f}")
        display(afficher_palette(palette, taille_case=40))
        display(im_r)

In [ ]:
# Test sur image 4 couleurs
comparer_methodes(im_4, 4)

# Test sur image générée avec distribution connue
# Image 100x100 avec 3 couleurs réparties inégalement
im_3 = Image.new('RGB', (100, 100))
px_3 = im_3.load()
setRegion(0, 0, 70, 100, (200, 50, 50), px_3)   # rouge
setRegion(70, 0, 20, 100, (50, 200, 50), px_3)  # vert
setRegion(90, 0, 10, 100, (50, 50, 200), px_3)  # bleu
comparer_methodes(im_3, 3)

# Test sur une image plus complexe
im = Image.open("lyon.png").convert("RGB")
comparer_methodes(im, 8)


7. Utilisez un pré-traitement des images (flou gaussien, etc) afin de lisser les couleurs. Cela est une piste afin de choisir de meilleurs couleurs représentatives. Proposez une comparaison de cette amélioration (ou de déterioration éventuelle) avec les autres méthodes.


In [ ]:
from PIL import Image, ImageFilter

def palette_avec_flou(im, k, rayon_flou=2):
    """
    Applique un flou gaussien PIL sur l'image, puis extrait une palette par k-means.
    Le flou est uniquement utilisé pour choisir la palette — l'image originale est re-coloriée.
    Arguments :
        im (Image): image PIL originale
        k (int): nombre de couleurs
        rayon_flou (int): rayon du flou gaussien
    Renvoie :
        tuple: (palette, im_re-coloriée)
    """
    # Appliquer le flou gaussien de PIL
    im_floue = im.filter(ImageFilter.GaussianBlur(radius=rayon_flou))
    freq_floue = lister_couleurs(im_floue)
    palette = palette_kmeans(freq_floue, k)
    im_recoloriee = recolorier(im, palette)  # Re-colorier l'originale
    return palette, im_recoloriee


def comparer_avec_sans_flou(im, k):
    """
    Compare la méthode k-means avec et sans pré-traitement par flou.
    Arguments :
        im (Image): image PIL
        k (int): nombre de couleurs
    """
    freq = lister_couleurs(im)
    print(f"\n=== Flou gaussien pour k={k} ===")
    
    # Sans flou
    p_sans_flou = palette_kmeans(freq, k)
    im_sans_flou = recolorier(im, p_sans_flou)
    e_sans_flou = score_erreur(im, im_sans_flou)
    print(f"Sans flou : erreur = {e_sans_flou:.4f}")
    display(afficher_palette(p_sans_flou))
    display(im_sans_flou)
    
    # Avec flou (différents rayons)
    for rayon in [1, 2, 5]:
        p_flou, im_flou = palette_avec_flou(im, k, rayon_flou=rayon)
        e_flou = score_erreur(im, im_flou)
        print(f"Avec flou r={rayon} : erreur = {e_flou:.4f}")
        display(afficher_palette(p_flou))
        display(im_flou)


# Test sur image 4 couleurs : le flou n'aide pas (les couleurs sont uniformes)
comparer_avec_sans_flou(im_4, 4)

# Test sur image 3 couleurs avec distribution inégale : le flou devrait aider à mieux regrouper les couleurs proches
comparer_avec_sans_flou(im_3, 3)

# Test sur une image plus complexe : le flou devrait aider à mieux regrouper les couleurs proches et réduire l'erreur
comparer_avec_sans_flou(im, 8)


8. Proposez une méthode d'amélioration de calcul de la distance entre deux couleurs, vous pouvez vous baser sur d'autres espaces de couleur [(doc)](https://fr.wikipedia.org/wiki/Espace_de_couleur). Cette partie est difficile, les espaces de couleurs possibles sont complexes à comprendre.



9. Optimisez les étapes précédentes (complexité, espace nécessaire, structures de données, etc.) et justifiez vos choix.


In [ ]:
def recolorier_optimise(im, palette):
    """
    Version améliorée de recolorier avec sauvegarde des correspondances couleur-palette.
    De nombreux pixels partagent la même couleur dans une image, on peut éviter
    de recalculer la couleur la plus proche pour chaque pixel.
    Arguments :
        im (Image): image PIL originale
        palette (list): liste de tuples (r, g, b)
    Renvoie :
        Image: nouvelle image re-coloriée
    """
    W, H = im.size
    im_out = Image.new('RGB', (W, H))
    px_in = im.load()
    px_out = im_out.load()
    cache = {} # {couleur_originale: couleur_palette}
    
    for x in range(W):
        for y in range(H):
            c = px_in[x, y]
            if c not in cache:
                cache[c] = couleur_la_plus_proche(c, palette)
            px_out[x, y] = cache[c]
    return im_out


# Vérification : même résultat que la version de départ
palette_test = palette_intervalles(freq_4, 4)
im_normal = recolorier(im_4, palette_test)
im_opti = recolorier_optimise(im_4, palette_test)

px_n = im_normal.load()
px_o = im_opti.load()
W, H = im_4.size
identiques = True
for x in range(W):
    for y in range(H):
        if px_n[x, y] != px_o[x, y]:
            identiques = False
            break
assert identiques, "Les deux versions doivent produire le même résultat"
print("Version optimisée produit le même résultat")

In [ ]:
# Mesure de temps
import time

# Image assez grande pour voir la différence
im_perf = Image.new('RGB', (200, 200))
px_perf = im_perf.load()
# Créer un dégradé simple pour avoir plus de couleurs uniques
for x in range(200):
    for y in range(200):
        r = (x * 255) // 200
        g = (y * 255) // 200
        b = 100
        px_perf[x, y] = r, g, b

freq_perf = lister_couleurs(im_perf)
palette_perf = palette_intervalles(freq_perf, 8)
print(f"Image test : {im_perf.size}, {len(freq_perf)} couleurs uniques")

t0 = time.time()
im_r1 = recolorier(im_perf, palette_perf)
t1 = time.time()
im_r2 = recolorier_optimise(im_perf, palette_perf)
t2 = time.time()

print(f"Version normale : {t1-t0:.3f}s")
print(f"Version optimisée : {t2-t1:.3f}s")

### Bonus

10. Créez une palette représentative à partir de plusieurs images.

In [ ]:
def fusionner_frequences(liste_freq):
    """
    Fusionne plusieurs dictionnaires de fréquences de couleurs en un seul.
    Les fréquences sont additionnées.
    Arguments :
        liste_freq (list): liste de dicts {couleur: fréquence}
    Renvoie :
        dict: dictionnaire fusionné {couleur: fréquence totale}
    """
    freq_total = {}
    for freq in liste_freq:
        for c, n in freq.items():
            if c in freq_total:
                freq_total[c] += n
            else:
                freq_total[c] = n
    return freq_total


def palette_multi_images(liste_images, k):
    """
    Crée une palette représentative à partir de plusieurs images.
    Fusionne les fréquences de couleurs de toutes les images,
    puis applique la méthode par k-means.
    Arguments :
        liste_images (list): liste d'images PIL
        k (int): nombre de couleurs
    Renvoie :
        list: palette de k couleurs représentative de toutes les images
    """
    # Lister les couleurs de chaque image
    liste_freq = [lister_couleurs(im) for im in liste_images]
    # Fusionner toutes les fréquences
    freq_total = fusionner_frequences(liste_freq)
    # Extraire la palette
    return palette_kmeans(freq_total, k)


# Test : deux images avec des couleurs différentes
# Image 1 : rouge/vert
im_a = Image.new('RGB', (100, 50))
px_a = im_a.load()
setRegion(0, 0, 50, 50, (200, 50, 50), px_a)
setRegion(50, 0, 50, 50, (50, 200, 50), px_a)

# Image 2 : bleu/jaune
im_b = Image.new('RGB', (100, 50))
px_b = im_b.load()
setRegion(0, 0, 50, 50, (50, 50, 200), px_b)
setRegion(50, 0, 50, 50, (200, 200, 50), px_b)

palette_multi = palette_multi_images([im_a, im_b], k=4)
print("Palette multi-images (k=4) :", palette_multi)
assert len(palette_multi) == 4
display(afficher_palette(palette_multi))

# Vérification : la palette doit représenter les deux images
e_a = score_erreur(im_a, recolorier_optimise(im_a, palette_multi))
e_b = score_erreur(im_b, recolorier_optimise(im_b, palette_multi))
print(f"Erreur sur image A : {e_a:.4f}")
print(f"Erreur sur image B : {e_b:.4f}")

# Comparer avec palette dédiée à chaque image
e_a_solo = score_erreur(im_a, recolorier_optimise(im_a, palette_kmeans(lister_couleurs(im_a), 4)))
e_b_solo = score_erreur(im_b, recolorier_optimise(im_b, palette_kmeans(lister_couleurs(im_b), 4)))
print(f"Erreur palette dédiée A : {e_a_solo:.4f}")
print(f"Erreur palette dédiée B : {e_b_solo:.4f}")
print("(La palette multi-images est un compromis entre les deux)")
print("Etape 10 réussie")